<a href="https://colab.research.google.com/github/Arsh-Vora/Arxiv-PageRank-Analysis/blob/main/PageRankAll.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pyspark kagglehub -q

import os
import time
import kagglehub
from pyspark.sql import SparkSession
from pyspark import SparkConf

print("Initializing Apache Spark...")

# Maximize memory for Colab's single-node environment
conf = SparkConf() \
    .setAppName("Bipartite_Spark_PageRank") \
    .setMaster("local[*]") \
    .set("spark.driver.memory", "8g") \
    .set("spark.executor.memory", "2g") \
    .set("spark.default.parallelism", "8")

spark = SparkSession.builder.config(conf=conf).getOrCreate()
sc = spark.sparkContext

print("Spark Context running. Downloading dataset...")
path = kagglehub.dataset_download("Cornell-University/arxiv")
json_path = os.path.join(path, "arxiv-metadata-oai-snapshot.json")
print("Dataset ready at:", json_path)

Initializing Apache Spark...
Spark Context running. Downloading dataset...


100%|██████████| 1.58G/1.58G [00:14<00:00, 116MB/s] 

Extracting files...


Dataset ready at: /root/.cache/kagglehub/datasets/Cornell-University/arxiv/versions/276/arxiv-metadata-oai-snapshot.json


### **Personal Note: Environment Setup**
Okay, first step is just getting Spark and the dataset. I had to set the `spark.driver.memory` to 8g because the default settings were causing the Java Virtual Machine to choke pretty early on. Also, using `local[*]` is key so Spark actually uses all the CPU cores in Colab.

* **What worked:** Increasing driver memory.
* **What didn't:** Standard settings (kept getting "GC overhead limit exceeded").
* **Warning:** If Colab disconnects, I have to re-run this and re-download the 1.5GB Arxiv file, which is a pain.

In [ ]:
import json

def parse_bipartite_edges(line):
    """
    Creates edges from Author -> Paper and Paper -> Author.
    This prevents the N^2 edge explosion of direct co-authorship.
    """
    try:
        record = json.loads(line)
        paper_id = f"P:{record.get('id')}"
        authors = record.get('authors_parsed', [])

        edges = []
        for a in authors:
            author_name = f"A:{a[0].strip()}, {a[1].strip()}"
            edges.append((author_name, paper_id))
            edges.append((paper_id, author_name))

        return edges
    except:
        return []

print("Mapping bipartite edges (100% of data, no filters)...")
raw_rdd = sc.textFile(json_path)

edges_rdd = raw_rdd.flatMap(parse_bipartite_edges).distinct()

Mapping bipartite edges (100% of data, no filters)...


### **Personal Note: Data Ingestion & Graph Logic**
This is where I fixed the huge crashing issue. Originally, I tried to link Author directly to Author (Unipartite), but that was a disaster. One paper with 3,000 authors created millions of edges and the system just died.

Instead, I’m doing **Author -> Paper -> Author**. It’s way more efficient because it’s linear instead of quadratic. 100% of the authors are kept here—no more deleting the big ATLAS or CMS collaborations just to save memory. Using `.distinct()` is slow but necessary so we don't count the same link twice.

In [ ]:
print("Building Graph and executing PageRank iterations...")
start_time = time.time()

links = edges_rdd.groupByKey().mapValues(list).cache()

ranks = links.mapValues(lambda v: 1.0)

def compute_contributions(urls_and_rank):
    """Calculates node contributions to the rank of its neighbors."""
    node, (neighbors, rank) = urls_and_rank
    num_neighbors = len(neighbors)
    for neighbor in neighbors:
        yield (neighbor, rank / num_neighbors)

iterations = 10
damping_factor = 0.85

for iteration in range(iterations):
    print(f"Running Spark Iteration {iteration + 1}/{iterations}...")

    contributions = links.join(ranks).flatMap(compute_contributions)

    from operator import add
    ranks = contributions.reduceByKey(add) \
                         .mapValues(lambda rank: (1 - damping_factor) + damping_factor * rank)

print(f"Distributed computation completed in {time.time() - start_time:.2f} seconds.")

Building Graph and executing PageRank iterations...
Running Spark Iteration 1/10...
Running Spark Iteration 2/10...
Running Spark Iteration 3/10...
Running Spark Iteration 4/10...
Running Spark Iteration 5/10...
Running Spark Iteration 6/10...
Running Spark Iteration 7/10...
Running Spark Iteration 8/10...
Running Spark Iteration 9/10...
Running Spark Iteration 10/10...
Distributed computation completed in 1.78 seconds.


### **Personal Note: Data Ingestion & Graph Logic**
This is where I fixed the huge crashing issue. Originally, I tried to link Author directly to Author (Unipartite), but that was a disaster. One paper with 3,000 authors created millions of edges and the system just died.

Instead, I’m doing **Author -> Paper -> Author**. It’s way more efficient because it’s linear instead of quadratic. 100% of the authors are kept here—no more deleting the big ATLAS or CMS collaborations just to save memory. Using `.distinct()` is slow but necessary so we don't count the same link twice.

In [ ]:
print("Triggering Spark DAG and extracting top authors...")

author_ranks = ranks.filter(lambda x: x[0].startswith("A:"))

top_20 = author_ranks.sortBy(lambda x: x[1], ascending=False).take(20)

print("\n--- Top 20 Authors by Bipartite Spark PageRank ---")
for i, (author, rank) in enumerate(top_20):
    clean_name = author[2:]
    print(f"{i+1}. {clean_name} (Score: {rank:.4f})")

spark.stop()

Triggering Spark DAG and extracting top authors...

--- Top 20 Authors by Bipartite Spark PageRank ---
1. CMS Collaboration,  (Score: 532.1447)
2. ATLAS Collaboration,  (Score: 498.9011)
3. Liu, Yang (Score: 357.1985)
4. Wang, Wei (Score: 305.5668)
5. Zhang, Wei (Score: 240.0899)
6. Taniguchi, Takashi (Score: 236.5448)
7. Watanabe, Kenji (Score: 227.3884)
8. Chen, Wei (Score: 213.3164)
9. Shelah, Saharon (Score: 208.9989)
10. Wang, Xin (Score: 207.4334)
11. Zhang, Lei (Score: 204.9867)
12. ALICE Collaboration,  (Score: 204.2677)
13. Zhang, Yu (Score: 202.4122)
14. Li, Yang (Score: 198.5796)
15. Wang, Hao (Score: 194.9463)
16. Liu, Wei (Score: 186.1436)
17. Li, Wei (Score: 183.4410)
18. Zhang, Rui (Score: 174.7560)
19. Wang, Jian (Score: 171.8789)
20. Chen, Xi (Score: 170.2287)


### **Personal Note: Results and Filtering**
Final step! Since the RDD has both "A:" (Authors) and "P:" (Papers), I have to filter it at the end to only see the authors.

The scores look huge (like 500+) because we started everyone at 1.0 instead of 1/N. This is exactly what was in the lecture slides. Seeing CMS and ATLAS at the top makes sense now—they have thousands of members, so they act like massive "hubs" in the network.

* **Takeaway:** This version is way better than the 86-second matrix version because it actually found Saharon Shelah and the big Physics groups properly.